# 06. 머신러닝 이진분류 - 실무전용 문제 모음

> **실무 전용 노트북** - 이론 설명 없이 TODO 문제만 모았습니다. (관련 이론 노트북: 06_머신러닝_분류_이진.ipynb)
이미 개념은 이해했다는 전제로, 손으로 더 많이 풀어보며 완전히 몸에 익히는 것이 목표입니다. 정답을 먼저 보지 말고 반드시 스스로 코드를 작성해본 뒤 펼쳐서 비교하세요.

이론 노트북에서는 Titanic으로 연습했다면, 여기서는 **마케팅 전환(구매여부) 데이터**로 이진분류를 다시 연습합니다.

## 목차
1. 모델 학습 연습
2. 평가지표 연습
3. confusion matrix & ROC curve 연습
4. 불균형 데이터 대응 연습

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/marketing_conversion.csv')
df['나이'] = df['나이'].fillna(df['나이'].mean())
df = pd.get_dummies(df, columns=['기기종류', '유입경로'], drop_first=True)
X = df.drop(columns=['구매여부'])
y = df['구매여부']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
X_train.shape

## 1. 모델 학습 연습

**문제 1.** `LogisticRegression(max_iter=1000)`을 학습시켜 예측값 `logi_pred`를 만드세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.linear_model import LogisticRegression
logi = LogisticRegression(max_iter=1000)  # max_iter를 늘려주지 않으면 데이터에 따라 수렴하지 못했다는 경고(ConvergenceWarning)가 뜰 수 있음
logi.fit(X_train_s, y_train)
logi_pred = logi.predict(X_test_s)
```

</details>

**문제 2.** `RandomForestClassifier(n_estimators=150, random_state=42)`를 학습시켜 예측값 `rf_pred`를 만드세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=150, random_state=42)  # 여러 트리를 배깅 방식으로 묶어 단일 트리보다 안정적인 성능을 냄
rf.fit(X_train_s, y_train)
rf_pred = rf.predict(X_test_s)
```

</details>

**문제 3.** `SVC(probability=True, random_state=42)`를 학습시켜 예측값 `svc_pred`를 만드세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.svm import SVC
svc = SVC(probability=True, random_state=42)  # probability=True를 줘야 이후 predict_proba()로 확률을 뽑을 수 있음(기본값 False)
svc.fit(X_train_s, y_train)
svc_pred = svc.predict(X_test_s)
```

</details>

## 2. 평가지표 연습

**문제 4.** 세 모델의 accuracy를 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, logi_pred), accuracy_score(y_test, rf_pred), accuracy_score(y_test, svc_pred))  # 여러 모델의 accuracy를 나란히 출력해 한눈에 비교
```

</details>

**문제 5.** `rf_pred`의 precision, recall, f1을 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import precision_score, recall_score, f1_score
print(precision_score(y_test, rf_pred), recall_score(y_test, rf_pred), f1_score(y_test, rf_pred))  # 이진분류에서는 average 옵션 없이 바로 계산됨(양성 클래스=1 기준)
```

</details>

**문제 6.** `rf`의 predict_proba로 roc_auc_score를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import roc_auc_score
proba = rf.predict_proba(X_test_s)[:, 1]  # roc_auc는 0/1이 아니라 '확률'을 입력으로 받음
print(roc_auc_score(y_test, proba))
```

</details>

## 3. confusion matrix & ROC curve 연습

**문제 7.** `rf_pred`의 confusion matrix를 heatmap으로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')  # annot=True로 숫자를, fmt='d'로 정수 형식을 지정
plt.show()
```

</details>

**문제 8.** `rf`의 ROC curve를 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, proba)  # thresholds는 여기서 쓰지 않으므로 _로 무시
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'k--')  # 대각선(랜덤 추측 수준)과 비교해 곡선이 왼쪽 위로 많이 벗어날수록 좋은 모델
plt.show()
```

</details>

## 4. 불균형 데이터 대응 연습

**문제 9.** `RandomForestClassifier(class_weight='balanced', random_state=42)`를 학습시켜 recall을 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import recall_score
rf_bal = RandomForestClassifier(class_weight='balanced', random_state=42)  # class_weight='balanced': 소수 클래스에 더 큰 가중치를 줘서 recall을 끌어올림
rf_bal.fit(X_train_s, y_train)
print(recall_score(y_test, rf_bal.predict(X_test_s)), recall_score(y_test, rf_pred))  # balanced 모델과 기본 모델의 recall을 나란히 비교
```

</details>

**문제 10.** `imblearn.SMOTE`로 오버샘플링한 뒤 `RandomForestClassifier`를 다시 학습시켜 recall을 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_over, y_over = smote.fit_resample(X_train_s, y_train)  # 학습 데이터에만 오버샘플링 적용(테스트 데이터는 그대로 둬야 실제 성능을 왜곡 없이 평가 가능)
rf_smote = RandomForestClassifier(random_state=42)
rf_smote.fit(X_over, y_over)
print(recall_score(y_test, rf_smote.predict(X_test_s)))
```

</details>